# Milestone 4.34: Handling Missing Values Using Drop and Fill Strategies

**Objective:** Master the critical skill of handling missing data responsibly using drop and fill strategies—making informed trade-offs that preserve data quality.

**Key Learning:**
- Missing data is inevitable in real-world datasets
- Handling it incorrectly can be worse than leaving it alone
- **Drop** = Remove incomplete data (lose information)
- **Fill** = Replace missing values (introduce assumptions)
- **Choice matters** - No automated solution fits all cases

**Think of it this way:** Handling missing data is like fixing a bridge—removing damaged sections makes it smaller, patching them keeps size but adds new material. Choose wisely.

---

## Why Missing Data Handling Matters

### The Problem

**Common beginner mistakes:**
- ❌ Dropping all rows with any missing data → Lose 90% of dataset
- ❌ Filling categorical data with mean → Creates nonsense ("Department: 3.5")
- ❌ Filling blindly without understanding context
- ❌ Hiding missing data problems instead of addressing them
- ❌ Making arbitrary choices without justification

**Result:** Analysis based on distorted or incomplete data → Wrong conclusions.

### The Solution

**Two main strategies:**

1. **Drop (Remove)**
   - **When:** Column is mostly empty OR row has critical data missing
   - **Effect:** Smaller dataset, but remaining data is real
   - **Risk:** Lose valuable information

2. **Fill (Impute)**
   - **When:** Column is important AND has few missing values
   - **Effect:** Full dataset, but some data is estimated
   - **Risk:** Introduce bias or dilute real patterns

**Professional habit:** Choose strategy based on context, not convenience.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

---

## Part 1: Understanding the Missing Data Problem

### Loading Data with Missing Values

In [ ]:
# Load survey data (has missing values)
survey_df = pd.read_csv('../data/raw/survey_data.csv')

print("Survey Data:")
print(survey_df)
print(f"\nShape: {survey_df.shape}")

### Step 1: Always Inspect Missing Data First

In [ ]:
# Check for missing values
print("Missing values per column:")
print(survey_df.isnull().sum())

print("\nMissing values percentage:")
missing_pct = (survey_df.isnull().sum() / len(survey_df)) * 100
print(missing_pct)

print("\nTotal missing values:", survey_df.isnull().sum().sum())

**Findings:**
- **Age:** 2 missing (20%)
- **Income:** 3 missing (30%)
- **Education:** 1 missing (10%)
- **Satisfaction:** 1 missing (10%)
- **City:** 2 missing (20%)

**Total:** 9 missing values across 10 rows → Many rows affected!

### Visualizing Missing Data Pattern

In [ ]:
# Show which rows have missing data
print("Rows with ANY missing values:")
print(survey_df[survey_df.isnull().any(axis=1)])

print(f"\nRows with missing data: {survey_df.isnull().any(axis=1).sum()} out of {len(survey_df)}")

**Key insight:** 7 out of 10 rows have at least one missing value!

**Implication:** If we drop all rows with ANY missing data, we lose 70% of the dataset.

---

## Part 2: Strategy 1 - Dropping Missing Values

### Method 1: Drop Rows with Any Missing Data

**Use when:** Row has critical missing data that can't be estimated

In [ ]:
# Drop ALL rows with ANY missing value
df_drop_any = survey_df.dropna()

print("After dropping rows with ANY missing value:")
print(df_drop_any)
print(f"\nOriginal shape: {survey_df.shape}")
print(f"After dropping: {df_drop_any.shape}")
print(f"Rows lost: {len(survey_df) - len(df_drop_any)} ({((len(survey_df) - len(df_drop_any)) / len(survey_df) * 100):.1f}%)")

**Result:** 10 rows → 3 rows (70% data loss!)

**⚠️ Warning:** This is often too aggressive!

**When to use:**
- Dataset is very large (millions of rows)
- Missing data is in critical columns
- No reasonable way to fill missing values

### Method 2: Drop Rows Only if ALL Values Are Missing

In [ ]:
# Drop rows where ALL values are missing
df_drop_all = survey_df.dropna(how='all')

print("After dropping rows with ALL missing values:")
print(f"Original shape: {survey_df.shape}")
print(f"After dropping: {df_drop_all.shape}")
print(f"Rows lost: {len(survey_df) - len(df_drop_all)}")

**Result:** No rows dropped (none are completely empty)

**When to use:**
- Remove completely empty rows
- Clean up data import artifacts

### Method 3: Drop Rows Based on Threshold

**Best practice:** Only drop rows with too many missing values

In [ ]:
# Drop rows with more than 2 missing values
# thresh = minimum number of NON-NULL values required
num_columns = len(survey_df.columns)
df_drop_thresh = survey_df.dropna(thresh=num_columns - 2)  # Allow max 2 missing

print("After dropping rows with more than 2 missing values:")
print(df_drop_thresh)
print(f"\nOriginal shape: {survey_df.shape}")
print(f"After dropping: {df_drop_thresh.shape}")
print(f"Rows lost: {len(survey_df) - len(df_drop_thresh)}")

**Result:** Much more reasonable—keeps most data!

**When to use:**
- When some missing values are acceptable
- Balance between data loss and quality

### Method 4: Drop Rows Based on Specific Columns

**Most common approach:** Drop only if critical columns are missing

In [ ]:
# Drop rows where Age or Income is missing (critical for analysis)
df_drop_subset = survey_df.dropna(subset=['Age', 'Income'])

print("After dropping rows with missing Age OR Income:")
print(df_drop_subset)
print(f"\nOriginal shape: {survey_df.shape}")
print(f"After dropping: {df_drop_subset.shape}")
print(f"Rows lost: {len(survey_df) - len(df_drop_subset)}")

**Result:** Targeted dropping based on business logic

**When to use:**
- Certain columns are essential for analysis
- Missing critical data makes row unusable
- **This is often the best approach!**

### Method 5: Drop Columns with Excessive Missing Data

In [ ]:
# Drop columns where >30% of values are missing
threshold = 0.3
missing_pct = survey_df.isnull().sum() / len(survey_df)
columns_to_drop = missing_pct[missing_pct > threshold].index

print(f"Columns with >{threshold*100:.0f}% missing data:")
print(columns_to_drop.tolist())

df_drop_cols = survey_df.drop(columns=columns_to_drop)

print(f"\nOriginal shape: {survey_df.shape}")
print(f"After dropping columns: {df_drop_cols.shape}")
print(f"Columns dropped: {len(columns_to_drop)}")

**When to drop columns:**
- Column has >50% missing data (unreliable)
- Column is not essential for analysis
- No reasonable way to fill missing values

**⚠️ Warning:** Don't drop important columns just because they have missing data!

---

## Part 3: Strategy 2 - Filling Missing Values

### Method 1: Fill with Constant Value

**Use for:** Categorical data or when you have domain knowledge

In [ ]:
# Fill missing City with 'Unknown'
df_fill_constant = survey_df.copy()
df_fill_constant['City'] = df_fill_constant['City'].fillna('Unknown')

print("After filling City with 'Unknown':")
print(df_fill_constant[['RespondentID', 'City']])
print(f"\nMissing values in City: {df_fill_constant['City'].isnull().sum()}")

**When to use constant fill:**
- Categorical data (text)
- You know a reasonable default (e.g., 'Unknown', 'Not Specified')
- Missing has meaning (e.g., 'No Response')

**Examples:**
- City → 'Unknown'
- Status → 'Pending'
- Response → 'No Answer'

### Method 2: Fill with Mean (Numeric Data)

In [ ]:
# Fill missing Income with mean income
df_fill_mean = survey_df.copy()
mean_income = df_fill_mean['Income'].mean()
df_fill_mean['Income'] = df_fill_mean['Income'].fillna(mean_income)

print(f"Mean income: ${mean_income:,.2f}")
print("\nAfter filling Income with mean:")
print(df_fill_mean[['RespondentID', 'Income']])
print(f"\nMissing values in Income: {df_fill_mean['Income'].isnull().sum()}")

**When to use mean fill:**
- Numeric data with normal distribution
- Few missing values (<10%)
- No extreme outliers

**⚠️ Warning:** Mean is sensitive to outliers!

### Method 3: Fill with Median (More Robust)

In [ ]:
# Fill missing Age with median age
df_fill_median = survey_df.copy()
median_age = df_fill_median['Age'].median()
df_fill_median['Age'] = df_fill_median['Age'].fillna(median_age)

print(f"Median age: {median_age}")
print("\nAfter filling Age with median:")
print(df_fill_median[['RespondentID', 'Age']])
print(f"\nMissing values in Age: {df_fill_median['Age'].isnull().sum()}")

**When to use median fill:**
- Numeric data with outliers
- Skewed distributions
- More robust than mean

**✅ Often better than mean!**

**Example:** Income data often has high earners (outliers) → Median is better

### Method 4: Fill with Mode (Most Common Value)

In [ ]:
# Fill missing Education with mode (most common education level)
df_fill_mode = survey_df.copy()
mode_education = df_fill_mode['Education'].mode()[0]  # mode() returns Series, get first value
df_fill_mode['Education'] = df_fill_mode['Education'].fillna(mode_education)

print(f"Most common education: {mode_education}")
print("\nAfter filling Education with mode:")
print(df_fill_mode[['RespondentID', 'Education']])
print(f"\nMissing values in Education: {df_fill_mode['Education'].isnull().sum()}")

**When to use mode fill:**
- Categorical data
- Discrete numeric data (e.g., ratings: 1, 2, 3, 4, 5)
- When you want most common value

**Examples:**
- Education level
- Department
- Rating score (1-5)

### Method 5: Forward Fill (Carry Previous Value Forward)

In [ ]:
# Forward fill - use previous row's value
df_ffill = survey_df.copy()
df_ffill['Satisfaction'] = df_ffill['Satisfaction'].fillna(method='ffill')

print("After forward filling Satisfaction:")
print(df_ffill[['RespondentID', 'Satisfaction']])
print(f"\nMissing values in Satisfaction: {df_ffill['Satisfaction'].isnull().sum()}")

**When to use forward fill:**
- Time series data (carry last known value)
- Sequential data where values change slowly
- Sensor readings, stock prices

**⚠️ Warning:** Not appropriate for independent survey responses!

### Method 6: Backward Fill (Carry Next Value Backward)

In [ ]:
# Backward fill - use next row's value
df_bfill = survey_df.copy()
df_bfill['Satisfaction'] = df_bfill['Satisfaction'].fillna(method='bfill')

print("After backward filling Satisfaction:")
print(df_bfill[['RespondentID', 'Satisfaction']])
print(f"\nMissing values in Satisfaction: {df_bfill['Satisfaction'].isnull().sum()}")

**When to use backward fill:**
- Similar to forward fill
- Time series where future value is known
- Less common than forward fill

### Method 7: Fill Multiple Columns at Once

In [ ]:
# Fill all numeric columns with their respective medians
df_fill_all = survey_df.copy()

numeric_columns = df_fill_all.select_dtypes(include=['number']).columns
for col in numeric_columns:
    median_val = df_fill_all[col].median()
    df_fill_all[col] = df_fill_all[col].fillna(median_val)
    print(f"Filled {col} with median: {median_val}")

print("\nMissing values after filling all numeric columns:")
print(df_fill_all.isnull().sum())

**Efficient approach:** Apply same strategy to multiple columns

---

## Part 4: Comparing Drop vs Fill Strategies

### Side-by-Side Comparison

In [ ]:
# Original data
print("=" * 70)
print("ORIGINAL DATA")
print("=" * 70)
print(f"Shape: {survey_df.shape}")
print(f"Missing values: {survey_df.isnull().sum().sum()}")
print("\nMissing per column:")
print(survey_df.isnull().sum())

In [ ]:
# Strategy 1: Drop all rows with ANY missing
df_strategy1 = survey_df.dropna()

print("\n" + "=" * 70)
print("STRATEGY 1: Drop ALL rows with ANY missing")
print("=" * 70)
print(f"Shape: {df_strategy1.shape}")
print(f"Rows lost: {len(survey_df) - len(df_strategy1)} ({((len(survey_df) - len(df_strategy1)) / len(survey_df) * 100):.1f}%)")
print(f"Missing values: {df_strategy1.isnull().sum().sum()}")
print("\n✅ Pros: Clean data, all real values")
print("❌ Cons: Lost 70% of data!")

In [ ]:
# Strategy 2: Drop only critical columns
df_strategy2 = survey_df.dropna(subset=['Age', 'Income'])

print("\n" + "=" * 70)
print("STRATEGY 2: Drop only if Age OR Income missing")
print("=" * 70)
print(f"Shape: {df_strategy2.shape}")
print(f"Rows lost: {len(survey_df) - len(df_strategy2)} ({((len(survey_df) - len(df_strategy2)) / len(survey_df) * 100):.1f}%)")
print(f"Missing values: {df_strategy2.isnull().sum().sum()}")
print("\n✅ Pros: Targeted approach, kept more data")
print("⚠️ Cons: Still has some missing values in other columns")

In [ ]:
# Strategy 3: Fill all missing values
df_strategy3 = survey_df.copy()
df_strategy3['Age'] = df_strategy3['Age'].fillna(df_strategy3['Age'].median())
df_strategy3['Income'] = df_strategy3['Income'].fillna(df_strategy3['Income'].median())
df_strategy3['Education'] = df_strategy3['Education'].fillna(df_strategy3['Education'].mode()[0])
df_strategy3['Satisfaction'] = df_strategy3['Satisfaction'].fillna(df_strategy3['Satisfaction'].median())
df_strategy3['City'] = df_strategy3['City'].fillna('Unknown')

print("\n" + "=" * 70)
print("STRATEGY 3: Fill all missing values")
print("=" * 70)
print(f"Shape: {df_strategy3.shape}")
print(f"Rows lost: 0 (0%)")
print(f"Missing values: {df_strategy3.isnull().sum().sum()}")
print("\n✅ Pros: Kept all data, no rows lost")
print("❌ Cons: Introduced assumptions, some values are estimates")

In [ ]:
# Strategy 4: Hybrid approach (Drop + Fill)
df_strategy4 = survey_df.copy()

# Drop rows where critical columns are missing
df_strategy4 = df_strategy4.dropna(subset=['Age', 'Income'])

# Fill remaining missing values in non-critical columns
df_strategy4['Education'] = df_strategy4['Education'].fillna(df_strategy4['Education'].mode()[0])
df_strategy4['Satisfaction'] = df_strategy4['Satisfaction'].fillna(df_strategy4['Satisfaction'].median())
df_strategy4['City'] = df_strategy4['City'].fillna('Unknown')

print("\n" + "=" * 70)
print("STRATEGY 4: HYBRID (Drop critical + Fill others)")
print("=" * 70)
print(f"Shape: {df_strategy4.shape}")
print(f"Rows lost: {len(survey_df) - len(df_strategy4)} ({((len(survey_df) - len(df_strategy4)) / len(survey_df) * 100):.1f}%)")
print(f"Missing values: {df_strategy4.isnull().sum().sum()}")
print("\n✅ Pros: Balanced approach - strict on critical, flexible on others")
print("✅ Best of both strategies!")

### Summary Table

In [ ]:
# Create comparison table
comparison = pd.DataFrame({
    'Strategy': ['Original', 'Drop All', 'Drop Critical', 'Fill All', 'Hybrid'],
    'Rows': [len(survey_df), len(df_strategy1), len(df_strategy2), len(df_strategy3), len(df_strategy4)],
    'Missing': [survey_df.isnull().sum().sum(), 
                df_strategy1.isnull().sum().sum(),
                df_strategy2.isnull().sum().sum(),
                df_strategy3.isnull().sum().sum(),
                df_strategy4.isnull().sum().sum()],
    'Data Lost %': [0, 
                    ((len(survey_df) - len(df_strategy1)) / len(survey_df) * 100),
                    ((len(survey_df) - len(df_strategy2)) / len(survey_df) * 100),
                    0,
                    ((len(survey_df) - len(df_strategy4)) / len(survey_df) * 100)]
})

print("\n" + "=" * 70)
print("COMPARISON OF STRATEGIES")
print("=" * 70)
print(comparison.to_string(index=False))
print("\n💡 Recommendation: Hybrid strategy often works best!")

---

## Part 5: Decision Framework - When to Drop vs Fill

### Decision Tree

```
Is the column critical for analysis?
├─ YES
│  ├─ How much data is missing?
│  │  ├─ <10% → FILL with median/mode
│  │  ├─ 10-30% → Consider DROPPING rows OR FILLING carefully
│  │  └─ >30% → DROP column (too unreliable)
│  └─
└─ NO (Optional column)
   ├─ How much data is missing?
   │  ├─ <20% → FILL with constant or statistics
   │  ├─ 20-50% → FILL with 'Unknown' or DROP column
   │  └─ >50% → DROP column
   └─
```

### Guidelines by Data Type

| Data Type | Missing % | Recommended Strategy |
|-----------|-----------|----------------------|
| **Numeric (Age, Income)** | <10% | Fill with **median** (robust to outliers) |
| | 10-30% | Drop rows OR fill with median |
| | >30% | Consider dropping column |
| **Categorical (City, Status)** | <20% | Fill with **'Unknown'** or **mode** |
| | 20-50% | Fill with 'Unknown' |
| | >50% | Drop column |
| **Ratings (1-5 scale)** | <10% | Fill with **median** or **mode** |
| | >10% | Consider dropping rows |
| **Time Series** | Any | **Forward fill** or **backward fill** |
| **Critical IDs** | Any | **Drop rows** (cannot estimate IDs) |

---

## Part 6: Avoiding Common Mistakes

### Mistake 1: Filling Categorical Data with Mean

In [ ]:
# ❌ BAD: Filling Education (categorical) with mean
# This creates nonsense like "Education: 2.3"

print("❌ WRONG APPROACH:")
print("Filling categorical 'Education' with mean makes no sense!")
print("\n✅ CORRECT APPROACH:")
print("Use mode (most common value) or 'Unknown' for categorical data")

### Mistake 2: Not Checking Impact After Changes

In [ ]:
# Always compare before and after
print("Before handling missing values:")
print(survey_df['Income'].describe())

df_after = survey_df.copy()
df_after['Income'] = df_after['Income'].fillna(df_after['Income'].median())

print("\nAfter filling with median:")
print(df_after['Income'].describe())

print("\n✅ Mean and distribution should be similar (not drastically changed)")

### Mistake 3: Dropping Too Aggressively

In [ ]:
# Check data loss percentage
original_rows = len(survey_df)
after_drop = len(survey_df.dropna())
loss_pct = ((original_rows - after_drop) / original_rows) * 100

print(f"Data loss: {loss_pct:.1f}%")

if loss_pct > 30:
    print("\n⚠️ WARNING: Losing >30% of data!")
    print("Consider filling instead of dropping")
else:
    print("\n✅ Data loss is acceptable")

### Mistake 4: Not Documenting Decisions

In [ ]:
# Good practice: Document your cleaning decisions
cleaning_log = {
    'Age': 'Filled with median (31.0) - 2 missing values',
    'Income': 'Filled with median (50500.0) - 3 missing values',
    'Education': 'Filled with mode (Bachelor) - 1 missing value',
    'Satisfaction': 'Filled with median (4.0) - 1 missing value',
    'City': 'Filled with constant (Unknown) - 2 missing values'
}

print("Cleaning Documentation:")
for col, decision in cleaning_log.items():
    print(f"  {col}: {decision}")

print("\n✅ Always document what you did and why!")

---

## Part 7: Real-World Example - Patient Records

### Loading and Inspecting Patient Data

In [ ]:
# Load patient records
patients_df = pd.read_csv('../data/raw/patient_records.csv')

print("Patient Records:")
print(patients_df)
print(f"\nShape: {patients_df.shape}")

In [ ]:
# Inspect missing data
print("Missing values analysis:")
missing_count = patients_df.isnull().sum()
missing_pct = (missing_count / len(patients_df)) * 100

missing_summary = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %': missing_pct
})
print(missing_summary[missing_summary['Missing Count'] > 0])

### Applying Hybrid Strategy

In [ ]:
# Step 1: Drop rows where PatientID or Name is missing (critical identifiers)
patients_clean = patients_df.dropna(subset=['PatientID', 'Name'])
print(f"After dropping rows with missing ID/Name: {patients_clean.shape}")

# Step 2: Fill Age with median
patients_clean['Age'] = patients_clean['Age'].fillna(patients_clean['Age'].median())
print(f"Filled Age with median: {patients_clean['Age'].median()}")

# Step 3: Fill Weight with median
patients_clean['Weight'] = patients_clean['Weight'].fillna(patients_clean['Weight'].median())
print(f"Filled Weight with median: {patients_clean['Weight'].median():.1f}")

# Step 4: Fill BloodPressure with mode (most common reading)
bp_mode = patients_clean['BloodPressure'].mode()[0] if len(patients_clean['BloodPressure'].mode()) > 0 else 'Unknown'
patients_clean['BloodPressure'] = patients_clean['BloodPressure'].fillna(bp_mode)
print(f"Filled BloodPressure with mode: {bp_mode}")

# Step 5: Fill Diagnosis with 'Unknown'
patients_clean['Diagnosis'] = patients_clean['Diagnosis'].fillna('Unknown')
print("Filled Diagnosis with 'Unknown'")

# Step 6: Fill LastVisit with 'Not Recorded'
patients_clean['LastVisit'] = patients_clean['LastVisit'].fillna('Not Recorded')
print("Filled LastVisit with 'Not Recorded'")

print("\n" + "=" * 70)
print(f"Original: {patients_df.shape} with {patients_df.isnull().sum().sum()} missing values")
print(f"Cleaned: {patients_clean.shape} with {patients_clean.isnull().sum().sum()} missing values")
print("=" * 70)

In [ ]:
# Verify cleaning
print("Cleaned Patient Records:")
print(patients_clean)

print("\n✅ All missing values handled!")
print(patients_clean.isnull().sum())

---

## Summary: Key Takeaways

### Drop vs Fill - Quick Reference

**DROP rows when:**
- ✅ Critical data is missing (can't estimate)
- ✅ Row is mostly empty (>50% missing)
- ✅ Dataset is very large (can afford data loss)
- ✅ Missing pattern is systematic (e.g., all incomplete surveys)

**FILL values when:**
- ✅ Few values missing (<10-20%)
- ✅ Column is important for analysis
- ✅ Dataset is small (can't afford to lose rows)
- ✅ You have reasonable estimate (mean, median, mode)

**DROP columns when:**
- ✅ Column has >50% missing data
- ✅ Column is not essential
- ✅ No reasonable way to fill

### Filling Strategies by Type

| Data Type | Best Strategy | Example |
|-----------|---------------|----------|
| **Numeric (continuous)** | Median | Age, Income, Weight |
| **Numeric (with outliers)** | Median | Salary, Price |
| **Categorical** | Mode or 'Unknown' | City, Department, Status |
| **Ratings (1-5)** | Median or Mode | Satisfaction, Rating |
| **Time Series** | Forward/Backward Fill | Stock prices, Temperature |
| **Identifiers** | DROP row | ID, Email, Username |

### The Hybrid Approach (Best Practice)

```python
# 1. Drop rows with missing critical columns
df = df.dropna(subset=['ID', 'CriticalField'])

# 2. Fill numeric columns with median
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Income'] = df['Income'].fillna(df['Income'].median())

# 3. Fill categorical with mode or constant
df['Category'] = df['Category'].fillna(df['Category'].mode()[0])
df['City'] = df['City'].fillna('Unknown')

# 4. Verify no missing values remain
print(df.isnull().sum())
```

### Critical Points to Remember

✅ **Always inspect missing data first** (`.isnull().sum()`)

✅ **Choose strategy based on context, not convenience**

✅ **Hybrid approach often works best** (Drop critical + Fill others)

✅ **Use median (not mean) for numeric data** (robust to outliers)

✅ **Never fill categorical with numeric values**

✅ **Check data loss percentage** (>30% is concerning)

✅ **Verify results after cleaning** (compare before/after)

✅ **Document your decisions** (what and why)

### Common Mistakes - Avoid These!

❌ Dropping all rows with ANY missing → Lose too much data

❌ Filling categorical data with mean → Nonsense values

❌ Not checking percentage of missing data → Blind decisions

❌ Using mean instead of median → Sensitive to outliers

❌ Not verifying after cleaning → Hidden issues remain

❌ Dropping important columns just because they have missing data

---

## Next Steps

1. **Practice on different datasets with missing values**
2. **Always inspect before handling**
3. **Choose strategy based on data type and context**
4. **Record your 2-minute demonstration video**
5. **Build the habit: Inspect → Decide → Handle → Verify**

**You're now ready to handle missing data responsibly!** 🎉